# phase 2 : préparer et construire le modèle.<br>
## Tâche 1 : découper en entraînement et test, dans l’ordre du temps


### importation de data et supprime de redoublant

In [40]:
from sklearn.preprocessing import StandardScaler
import numpy as np 
import pandas as pd

In [ ]:
df=pd.read_csv("../data/raw/creditcard.csv")
df.duplicated().sum()
df_clean=df.drop_duplicates()
df_clean.duplicated().sum()

np.int64(0)

### decoupage chronologique


In [ ]:
data=df_clean.sort_values("Time")
copure=int(len(data)*0.8) 
train=data.iloc[:copure] # pour que 80% des donnes de entrainement 
test=data.iloc[copure:] #pour que 20% de donner de test
train["Class"].sum()+ test["Class"].sum()

np.int64(473)

In [18]:
len(test)+len(train)

283726

In [19]:
print(train["Time"].max() <= test["Time"].min())

True


## tâche 2 : mettre Amount à l’échelle, sans fuite de données
`1. Le contexte`

Les colonnes V1 à V28 sont déjà sur des échelles comparables (elles viennent d’une ACP). Amount, elle, va de 0 à 25 691 avec une médiane de 22 : quelques gros montants écrasent tout le reste. Dans Time, on a décidé de ne pas utiliser le brut.

In [32]:
df_clean["Amount"].describe()

count    283726.000000
mean         88.472687
std         250.399437
min           0.000000
25%           5.600000
50%          22.000000
75%          77.510000
max       25691.160000
Name: Amount, dtype: float64

moyenne tres superieur a la mediane (50%) :sine d'étirement  88>>>22

In [ ]:
df_clean["Amount"].skew()   

np.float64(16.978803370060476)

coffecient d'asymetrie : 16>>1 tres étiré 

In [ ]:
# 1. l'algorihtme: sans apprentissage donc sur train et test 
train["Amount_log"] = np.log1p(train["Amount"])
test["Amount_log"] = np.log1p(test["Amount"])


In [ ]:
# 2.standarisation : fit sur train seulement 
scaler=StandardScaler()
train["Amount_s"] = scaler.fit_transform(train[["Amount_log"]])
test["Amount_s"]=scaler.transform(test[["Amount_log"]])
train[["Amount","Amount_log","Amount_s"]].head(15)
#La valeur standardisée se lit comme un nombre d’écarts-types par rapport à la moyenne (du train, en échelle logarithmique) :

#0 : montant dans la moyenne ;
#positif : au-dessus de la moyenne (ligne 0, 149,62 → +1,10) ;
#négatif : en dessous (ligne 1, 2,69 → −1,13) ;
#au-delà de ±3 : valeur rare ou extrême.

,Amount,Amount_log,Amount_s
0,149.62,5.014760,1.099723
1,2.69,1.305626,-1.134338
2,378.66,5.939276,1.656571
3,123.50,4.824306,0.985010
4,69.99,4.262539,0.646650
5,3.67,1.541159,-0.992474
6,4.99,1.790091,-0.842538
7,40.80,3.732896,0.327639
8,93.20,4.545420,0.817033
9,3.68,1.543298,-0.991185


In [58]:
train["Amount_s"].describe()

count    2.269800e+05
mean     2.444232e-16
std      1.000002e+00
min     -1.920734e+00
25%     -7.486866e-01
50%      5.623500e-03
75%      7.261014e-01
max      4.033866e+00
Name: Amount_s, dtype: float64

In [59]:
print(train["Amount"].skew(), train["Amount_log"].skew())

14.069984483679008 0.13559041406977693


##### Tâche 3 : séparer les variables (X) et la cible (y)

X : les informations qu’il utilise pour prédire (les colonnes).<br>
y : la réponse à retrouver (ici Class, 0 normale ou 1 fraude).<br>
* La cible n’est jamais dans X. Sinon le modèle « lit la réponse » et obtient un score parfait, qui ne vaut rien en vrai. C’est une fuite de données.
* On retire les versions brutes remplacées. Amount et Amount_log ont été transformées en Amount_s : on garde seulement Amount_s, pour ne pas dupliquer l’information.
* On retire Time. Il ne se répète pas d’un jour à l’autre (décision de l’étape 9). On l’a gardé uniquement pour découper.
* Hour est une variable candidate. On va la préparer à part, pour comparer le modèle avec et sans elle.

In [65]:
colonnes=[f"V{i}" for i in range (1,29)]+["Amount_s"]
X_train=train[colonnes]
Y_train=train["Class"]
X_test=test[colonnes]
Y_test=test["Class"]


In [ ]:
# controles
list(X_train.columns) == list(X_test.columns)
Y_train.sum() + Y_test.sum()
X_train.isna().sum().sum()
X_train.shape[1]

29

#### Tâche 4 : entraîner un modèle de base et voir pourquoi l’accuracy trompe
* 1. Le contexte

Un modèle de base (baseline) est le modèle le plus simple possible. Il sert de point de comparaison : tout modèle plus compliqué doit faire mieux que lui, sinon il ne sert à rien. On commence par la régression logistique, qui calcule une probabilité de fraude à partir des colonnes de X.

* 2. Le pourquoi : pourquoi le score le plus connu ne suffit pas

L’accuracy est la proportion de prédictions correctes. Avec 0,17 % de fraudes, un « modèle » qui répond toujours « normale » se trompe seulement sur les fraudes : son accuracy est d’environ 99,8 %, alors qu’il ne détecte aucune fraude. Un score élevé peut donc cacher un modèle inutile. Voilà pourquoi on calcule l’accuracy ici, pour la comparer à ce « modèle qui répond toujours 0 », puis on utilisera de meilleures mesures à la tâche 5.

In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

modele = LogisticRegression(max_iter=1000)   # 1. créer
modele.fit(X_train, Y_train)                 # 2. apprendre sur le train
Y_pred = modele.predict(X_test)              # 3. prédire sur le test

accuracy_score(Y_test, Y_pred)*100            # proportion de bonnes prédictions



99.93303492757198

In [88]:
Y_zero = np.zeros(len(Y_test))               # modèle idiot : toujours 0
accuracy_score(Y_test, Y_zero)*100

99.86959433264019

In [86]:
print("nbr de prdections predit",Y_pred.sum())
print("nbr de predection d'origine ",Y_test.sum())

nbr de prdections predit 46
nbr de predection d'origine  74
